# E3 · Completar los modelos de E2 a diez semillas

La **CNN-LSTM con atención aditiva ya completó sus diez semillas**. Este cuaderno completa las cuatro semillas faltantes de cada una de las trece redes de E2: **52 ajustes nuevos**, con sus recetas originales. Conserva los seis resultados previos de cada red y las tres referencias deterministas.

1. En **Entorno de ejecución → Cambiar tipo de entorno**, selecciona una GPU. Los tiempos de referencia corresponden a una **A100**.
2. En **Secretos**, habilita para este cuaderno `GITHUB_TOKEN` con permiso de escritura del contenido del repositorio `jassan-bit/DeepLearning_202630`. No pegues el token en el código.
3. Ejecuta la celda siguiente. Guarda el progreso en **GitHub**, en la rama `e2-extension-10-semillas-v1`; no utiliza Drive.

El entrenamiento se estima en unos **64 minutos en A100**. La instalación, la evaluación y la publicación añaden tiempo. Esta etapa tiene un límite separado de **7200 segundos de procesos científicos**, incluidos fallos; mantiene cerrado el contador anterior de E3.

Se confirman en GitHub los **52 ajustes y calibradores antes de generar nuevas predicciones de prueba**. El resultado combina 133 predictores de E2 con las diez semillas ya publicadas de E3 sobre las mismas 608 fechas. Es una comparación descriptiva posterior: E2 usa 655 ejemplos de entrenamiento y E3 usó 532. No realiza pruebas de hipótesis ni acredita R1/R3.

Si falla una publicación después de completar un ajuste, ejecuta esta misma celda para publicar el cálculo conservado y continuar. Si Colab se interrumpe dentro de un ajuste, el entrenador original conserva ese caso y detiene la etapa para revisarlo: no reinicia automáticamente un entrenamiento parcial.


In [ ]:
# Ejecuta esta celda para preparar y continuar la extensión fija de E2.
from pathlib import Path
from urllib.request import Request, urlopen
import hashlib
import importlib.util
import os
import sys
import time
import uuid

sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

REPOSITORIO_E2 = "jassan-bit/DeepLearning_202630"
COMMIT_CODIGO_E2 = "4f4fa4687ce588c93436b6d714d6dc8bb5bac66e"
ARCHIVOS_E2 = {
    "bootstrap.py": "aa3d13a470c080cae1d99bca6f92944d9f8a1f1c0c196b9c91ef55b8b48f2c43",
    "prepare.py": "cb5b03de135797a1aa6cb39dbfd33743c3353455aae0cf41f027479bd542929b",
    "worker.py": "e3ef14469ec804616f9f05d30633d50ba3ca8351c70bd82a99d53e15687b6285",
    "aggregate.py": "15cd5107eaa57ecca0b9bf842bf22b2c742ac6c3bf4004720ff6a2f0dc13aac4",
    "plan.json": "45fb6a14137b03642737521731c4b56a1312980bfbeca0319182a9ffd600cc07"
}
CARPETA_E2 = Path("/content/e2_extension_10_semillas_v1/launcher")
CARPETA_E2.mkdir(parents=True, exist_ok=True)

def descargar_codigo_e2(nombre, esperado):
    destino = CARPETA_E2 / nombre
    if destino.is_symlink():
        raise RuntimeError("La ruta del código debe ser un archivo regular.")
    if destino.exists():
        if not destino.is_file() or hashlib.sha256(destino.read_bytes()).hexdigest() != esperado:
            raise RuntimeError(f"El archivo local {nombre} difiere del código fijado; se conserva para revisión.")
        return
    url = (f"https://raw.githubusercontent.com/{REPOSITORIO_E2}/{COMMIT_CODIGO_E2}/"
           f"experiments/e2_extension_10_semillas_v1/{nombre}")
    for intento in range(3):
        try:
            solicitud = Request(url, headers={"User-Agent": "E2-extension-Colab"})
            with urlopen(solicitud, timeout=60) as respuesta:
                contenido = respuesta.read(2 * 1024 * 1024 + 1)
            if len(contenido) > 2 * 1024 * 1024 or hashlib.sha256(contenido).hexdigest() != esperado:
                raise RuntimeError(f"SHA256 incorrecto para {nombre}.")
            temporal = destino.with_name(destino.name + "." + uuid.uuid4().hex + ".tmp")
            with temporal.open("xb") as salida:
                salida.write(contenido)
            temporal.replace(destino)
            return
        except (TimeoutError, ConnectionError, OSError):
            if intento == 2:
                raise RuntimeError(f"No se pudo descargar {nombre}; todavía no se inició ningún ajuste.") from None
            time.sleep(intento + 1)

for nombre_e2, sha256_e2 in ARCHIVOS_E2.items():
    descargar_codigo_e2(nombre_e2, sha256_e2)

print("Código verificado:", COMMIT_CODIGO_E2)
print("E2: 52 ajustes adicionales; CNN-LSTM con atención: se reutilizan sus diez semillas terminadas.")
especificacion_e2 = importlib.util.spec_from_file_location(
    "_e2_extension_" + uuid.uuid4().hex, CARPETA_E2 / "bootstrap.py"
)
modulo_e2 = importlib.util.module_from_spec(especificacion_e2)
especificacion_e2.loader.exec_module(modulo_e2)
RESULTADO_EXTENSION_E2 = modulo_e2.run()
